In [1]:
import tkinter as tk
from tkinter import ttk, scrolledtext
import pandas as pd


# Load prediction results
df = pd.read_csv("prediction_results.csv")

label_map = {
    "CG": "Computer Generated Review",
    "OR": "Original Review"
}

current_index = 0


# Colours
BG = "#F4F7FC"
CARD = "#FFFFFF"
BLUE = "#1565C0"
GREEN = "#2E7D32"
RED = "#C62828"
TEXT = "#222222"


# Functions
def load_review(index):
    global current_index

    if index < 0 or index >= len(df):
        return

    current_index = index

    row = df.iloc[index]

    txt_review.config(state="normal")
    txt_review.delete("1.0", tk.END)
    txt_review.insert(tk.END, row["text_"])
    txt_review.config(state="disabled")

    entry_index.delete(0, tk.END)
    entry_index.insert(0, str(index))

    lbl_progress.config(
        text=f"Review {index+1} of {len(df)}"
    )

    lbl_category.config(text=f"Category : {row['category']}")
    lbl_rating.config(text=f"Rating : {row['rating']}")

    actual = label_map.get(row["label"], row["label"])
    pred = label_map.get(row["Predicted"], row["Predicted"])

    lbl_actual.config(
        text=f"Actual Review : {actual}"
    )

    lbl_pred.config(
        text=f"Predicted Review : {pred}"
    )

    if bool(row["Correct"]):

        lbl_status.config(
            text="Prediction Correct ✓",
            bg=GREEN,
            fg="white"
        )

    else:

        lbl_status.config(
            text="Prediction Incorrect ✗",
            bg=RED,
            fg="white"
        )


def next_review():
    if current_index < len(df)-1:
        load_review(current_index+1)


def previous_review():
    if current_index > 0:
        load_review(current_index-1)


def goto_review():
    try:
        idx = int(entry_index.get())
        load_review(idx)
    except:
        pass


def reset():
    load_review(0)


# Window
root = tk.Tk()
root.title("Fake Review Detection Prototype")
root.geometry("1100x760")
root.configure(bg=BG)

title = tk.Label(
    root,
    text="Fake Review Detection Prototype",
    bg=BLUE,
    fg="white",
    font=("Arial",22,"bold"),
    pady=12
)
title.pack(fill="x")

stats = tk.Frame(root,bg=BG)
stats.pack(fill="x",pady=10)

accuracy = df["Correct"].mean()*100

tk.Label(
    stats,
    text=f"Dataset Size : {len(df)}",
    bg=BG,
    fg=TEXT,
    font=("Arial",12,"bold")
).pack(side="left",padx=20)

tk.Label(
    stats,
    text=f"Accuracy : {accuracy:.2f}%",
    bg=BG,
    fg=GREEN,
    font=("Arial",12,"bold")
).pack(side="left",padx=20)

lbl_progress = tk.Label(
    stats,
    text="",
    bg=BG,
    fg=BLUE,
    font=("Arial",12,"bold")
)
lbl_progress.pack(side="right",padx=20)

nav = tk.Frame(root,bg=BG)
nav.pack()

tk.Label(nav,text="Row :",bg=BG,font=("Arial",11)).grid(row=0,column=0,padx=5)

entry_index = ttk.Entry(nav,width=8)
entry_index.grid(row=0,column=1)

ttk.Button(nav,text="Go",command=goto_review).grid(row=0,column=2,padx=5)
ttk.Button(nav,text="◀ Previous",command=previous_review).grid(row=0,column=3,padx=5)
ttk.Button(nav,text="Next ▶",command=next_review).grid(row=0,column=4,padx=5)
ttk.Button(nav,text="Reset",command=reset).grid(row=0,column=5,padx=5)
ttk.Button(nav,text="Exit",command=root.destroy).grid(row=0,column=6,padx=5)

card = tk.Frame(root,bg=CARD,bd=2,relief="groove")
card.pack(fill="both",expand=True,padx=20,pady=20)

lbl_category = tk.Label(card,text="",bg=CARD,font=("Arial",12,"bold"))
lbl_category.pack(anchor="w",padx=15,pady=(15,2))

lbl_rating = tk.Label(card,text="",bg=CARD,font=("Arial",12,"bold"))
lbl_rating.pack(anchor="w",padx=15)

tk.Label(
    card,
    text="Original Review",
    bg=CARD,
    fg=BLUE,
    font=("Arial",14,"bold")
).pack(anchor="w",padx=15,pady=(15,5))

txt_review = scrolledtext.ScrolledText(
    card,
    width=120,
    height=16,
    font=("Arial",11),
    wrap=tk.WORD
)
txt_review.pack(padx=15)
txt_review.config(state="disabled")

lbl_actual = tk.Label(card,text="",bg=CARD,font=("Arial",13,"bold"))
lbl_actual.pack(anchor="w",padx=15,pady=(15,2))

lbl_pred = tk.Label(card,text="",bg=CARD,font=("Arial",13,"bold"))
lbl_pred.pack(anchor="w",padx=15,pady=(2,10))

lbl_status = tk.Label(
    card,
    text="",
    font=("Arial",16,"bold"),
    padx=15,
    pady=8
)
lbl_status.pack(pady=10)

load_review(0)

root.mainloop()
